# Youtube Download

In [4]:
import yt_dlp

def download_video(song_name):
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': 'input_video.%(ext)s',
        'default_search': 'ytsearch1:',  # 搜索并取第一个结果
        'noplaylist': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([song_name])
    return "input_video.mp4"

In [5]:
download_video("Poor thang")

[generic] Extracting URL: Poor thang
[youtube:search] Extracting URL: ytsearch1:Poor thang
[download] Downloading playlist: Poor thang
[youtube:search] query "Poor thang": Downloading web client config
[youtube:search] query "Poor thang" page 1: Downloading API JSON
[youtube:search] Playlist Poor thang: Downloading 1 items of 1
[download] Downloading item 1 of 1
[youtube] Extracting URL: https://www.youtube.com/watch?v=3aIjM1YGOO8
[youtube] 3aIjM1YGOO8: Downloading webpage


[youtube] 3aIjM1YGOO8: Downloading android vr player API JSON
[info] 3aIjM1YGOO8: Downloading 1 format(s): 399+140
[download] Destination: input_video.f399.mp4
[download] 100% of   17.72MiB in 00:00:05 at 3.46MiB/s   
[download] Destination: input_video.f140.m4a
[download] 100% of    4.48MiB in 00:00:02 at 2.20MiB/s   
[Merger] Merging formats into "input_video.mp4"
Deleting original file input_video.f140.m4a (pass -k to keep)
Deleting original file input_video.f399.mp4 (pass -k to keep)
[download] Finished downloading playlist: Poor thang


'input_video.mp4'

# Music track extraction
## fast-whisper
句子不完整

In [ ]:
import os
import subprocess
from faster_whisper import WhisperModel

def extract_and_transcribe(video_path):
    audio_path = "temp_audio.wav"
    
    # --- 步骤 1: 使用 FFmpeg 提取标准音频 ---
    print("正在提取音轨...")
    subprocess.run([
        'ffmpeg', '-i', video_path, 
        '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', 
        audio_path, '-y'
    ], check=True)

    # --- 步骤 2: 加载 Whisper 模型 ---
    # Mac M1/M2/M3 用户建议使用 cpu 模式（faster-whisper 对 MPS 支持在持续更新，CPU 已经够快）
    model_size = "medium" # 嘻哈建议 medium 起步，large-v3 效果最好但慢一点
    model = WhisperModel(model_size, device="cpu", compute_type="int8")

    print(f"正在识别歌词 (模型: {model_size})...")
    segments, info = model.transcribe(
        audio_path,
        # 1. 简化的 Prompt：只给关键词，不给完整句子
        initial_prompt="Rap, Hip-hop, lyrics, slang.", 
        
        # 2. 提高采样门槛：如果 AI 不确定，就不要乱说话
        beam_size=5,
        best_of=5,
        
        # 3. 关键：设置静音过滤和压缩比限制
        vad_filter=True,  # 确保开启静音过滤
        vad_parameters=dict(min_silence_duration_ms=500), # 超过0.5秒没声音就跳过
        
        # 4. 幻听控制：如果这一段重复率太高，丢弃它
        compression_ratio_threshold=2.4, 
        no_speech_threshold=0.6,
        
        word_timestamps=True
    )

    # --- 步骤 3: 格式化为 SRT 结构 ---
    # results = []
    # for segment in segments:
    #     results.append({
    #         'start': segment.start,
    #         'end': segment.end,
    #         'text': segment.text.strip()
    #     })
    #     print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

    # # 清理临时文件
    # os.remove(audio_path)
    # return results
    # --- 3. 核心修改：提取并临时存储 ---
    # 存储完整信息的列表（包含时间戳，用于最后生成 SRT）
    full_data = []
    # 纯英文文本列表（用于交给翻译引擎）
    english_texts_only = []

    for segment in segments:
        clean_text = segment.text.strip()
        if clean_text:
            # 存储详细信息
            full_data.append({
                'start': segment.start,
                'end': segment.end,
                'text': clean_text
            })
            # 存储纯文本
            english_texts_only.append(clean_text)
            # print(f"已提取: {clean_text}")

    # 清理临时音频
    if os.path.exists(audio_path):
        os.remove(audio_path)

    # 返回这两个列表供后续使用
    return full_data, english_texts_only

# --- 使用示例 ---
data_with_time, pure_texts = extract_and_transcribe("input_video.mp4")
print(pure_texts)

# 现在你可以直接把 pure_texts 扔给任何翻译工具了
# print(f"待翻译总句数: {len(pure_texts)}")


/Users/randy/Library/Mobile Documents/com~apple~CloudDocs/Document/project/randyTranslation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在提取音轨...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'input_video.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomav01iso2mp41
    encoder         : Lavf62.3.100
  Du

正在识别歌词 (模型: medium)...
['Young puss playing war games, He wanted love, but he only made more pain.', 'Poor thing, young puss playing war games, He wanted love, but he only made more pain.', 'Picture my soul, climbing out an infinite hole, Where niggas die over pride and live for the dough.', "If I survive, then I'll strive to hit with the flow, Hoping these waves are paid, but the ripples are slow.", "And I'm conflicted with no digits to show, But fantasy's a frivolous hose, gripping this pole,", "The temperature's low, but you know that ain't stopping, Little nigga for sliding like the tides on a whip in the snow.", 'Vicious for show, a tip torn around the abyss, With sticks blowing, piss pulled with attention to clips,', 'The wrist glowing, face tits from rent owing, Fist clenched when niggas diss, but since knowing,', "Lives expire when niggas' pride gets hired, In the vocal range, inside the rise vocal chain.", "My road to fame is rife with spikes and broken lanes, And tolls I can'

## WhisperX
句子完整但很长

In [ ]:
import whisperx
import gc

def process_mv_with_whisperx(video_path, device="cpu"):
    # 1. 加载模型 (针对 Mac M1/M2/M3 优化)
    model = whisperx.load_model("medium", device, compute_type="int8")
    
    # 2. 直接传视频路径，whisperx 会自动提取音频流进行转录
    print(f"开始转录视频内容: {video_path}")
    result = model.transcribe(video_path, batch_size=16) 
    
    # 3. 对齐 (这是解决你“断句碎裂”的关键)
    # 它会加载 wav2vec2 模型，精准对齐每个单词的时间点
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
    aligned_result = whisperx.align(result["segments"], model_a, metadata, video_path, device, return_char_alignments=False)
    
    # 4. 语义合并 (解决歌词碎裂的逻辑)
    # WhisperX 提供了更加连贯的 segments，但我们还可以手动根据停顿（Gap）进一步优化
    final_segments = merge_broken_sentences(aligned_result["segments"])
    
    return final_segments

def merge_broken_sentences(segments, min_pause=0.8):
    """
    如果两段歌词之间的停顿小于 min_pause 秒，且上一段没有结束标点，则合并。
    """
    merged = []
    if not segments: return merged
    
    curr = segments[0]
    for i in range(1, len(segments)):
        nxt = segments[i]
        # 判断：如果两段间隔很短（比如 0.5秒内），大概率是同一句歌词换气
        gap = nxt['start'] - curr['end']
        
        if gap < min_pause:
            curr['text'] = curr['text'].strip() + " " + nxt['text'].strip()
            curr['end'] = nxt['end']
        else:
            merged.append(curr)
            curr = nxt
    merged.append(curr)
    return merged

# AI translation

In [3]:
# !pip install llama-cpp-python

from llama_cpp import Llama

# 修改初始化代码
llm = Llama.from_pretrained(
    repo_id="Randyliu99/qwen2.5-7b-jcole-gguf",
    filename="Qwen2.5-7B-Instruct.Q4_K_M.gguf",
    n_ctx=2048,  # 设置为 2048 或更高，解决 1040 报错
    n_gpu_layers=-1 # 如果有显卡/金属加速，记得开启
)

def generate_bilingual_srt(full_data, english_texts, output_file="final_bilingual.srt"):
    """
    full_data: 包含 start, end, text 的列表
    english_texts: 纯英文文本列表
    """
    
    # --- 步骤 A: 构建 Prompt ---
    # 将歌词合并，并带上序号，方便模型对应
    lyrics_block = "\n".join([f"{i+1}: {text}" for i, text in enumerate(english_texts)])
    
    prompt = f"""你是一个专业的 Hip-hop 翻译官。请将以下歌词翻译成中文。
要求：
1. 保持行数和序号一一对应。
2. 翻译要地道，保留 Rap 的俚语和韵味。
3. 只返回翻译后的中文，不要包含任何解释。
4. 语义通顺并能理解上下文含义。
5. 根据歌词的内容设定语境。
6. 不含脏话，敏感词汇会进行隐晦处理。

歌词列表：
{lyrics_block}
"""

    # --- 步骤 B: 调用本地模型翻译 ---
    print("正在调用本地 Qwen 模型进行语境翻译...")
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048, # 歌词长的话需要调大
        temperature=0.7
    )
    
    translated_content = response['choices'][0]['message']['content']
    
    # 解析模型返回的中文（假设模型按行返回）
    # 简单的清理逻辑：去掉序号，只留文本
    chinese_lines = []
    for line in translated_content.strip().split('\n'):
        # 去掉类似 "1: " 或 "1. " 的前缀
        import re
        clean_line = re.sub(r'^\d+[:.、\s]+', '', line)
        chinese_lines.append(clean_line)

    # --- 步骤 C: 组合生成 SRT ---
    print(f"写入双语字幕到 {output_file}...")
    with open(output_file, "w", encoding="utf-8") as f:
        for i, (item, cn_text) in enumerate(zip(full_data, chinese_lines), start=1):
            start_str = format_timestamp(item['start'])
            end_str = format_timestamp(item['end'])
            
            f.write(f"{i}\n")
            f.write(f"{start_str} --> {end_str}\n")
            f.write(f"{item['text']}\n")  # 英文原文
            f.write(f"{cn_text}\n\n")    # 中文译文

    print("✅ 所有任务已完成！")

# --- 辅助函数：时间格式化 ---
def format_timestamp(seconds: float):
    millis = int((seconds - int(seconds)) * 1000)
    td_sec = int(seconds)
    td_min, td_sec = divmod(td_sec, 60)
    td_hour, td_min = divmod(td_min, 60)
    return f"{td_hour:02}:{td_min:02}:{td_sec:02},{millis:03}"

# --- 运行流程 ---
# 1. 运行你之前的 Whisper 代码拿到 data_with_time 和 pure_texts
# 2. 调用本函数：
# generate_bilingual_srt(data_with_time, pure_texts)
generate_bilingual_srt(data_with_time, pure_texts)


llama_model_load_from_file_impl: using device Metal (Apple M4) - 12118 MiB free
llama_model_loader: loaded meta data with 26 key-value pairs and 339 tensors from /Users/randy/.cache/huggingface/hub/models--Randyliu99--qwen2.5-7b-jcole-gguf/snapshots/797a41ca8b8b3b1335a08e43e3b4f1bb52bb1ae3/./Qwen2.5-7B-Instruct.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Unsloth_Gguf_M7Jqshyv
llama_model_loader: - kv   3:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   4:                         general.size_label str              = 7.6B
llama_model_loader: - kv   5:              

正在调用本地 Qwen 模型进行语境翻译...


llama_perf_context_print:        load time =    5546.20 ms
llama_perf_context_print: prompt eval time =    5545.51 ms /  1040 tokens (    5.33 ms per token,   187.54 tokens per second)
llama_perf_context_print:        eval time =   55968.15 ms /   957 runs   (   58.48 ms per token,    17.10 tokens per second)
llama_perf_context_print:       total time =   62655.03 ms /  1997 tokens
llama_perf_context_print:    graphs reused =        926


写入双语字幕到 final_bilingual.srt...
✅ 所有任务已完成！


# 生成中英字幕视频

In [8]:
import os
import subprocess
import shutil

def merge_subtitles_to_video(video_path, srt_path, output_filename="final_rap_video.mp4"):
    """
    使用带 libass 的 FFmpeg 压制双语字幕
    """
    # 1. 确保使用绝对路径，避免 iCloud 干扰
    abs_video = os.path.abspath(video_path)
    abs_srt = os.path.abspath(srt_path)
    
    # 2. 定义输出路径（默认放在原视频同目录下）
    output_path = os.path.join(os.path.dirname(abs_video), output_filename)

    # 3. 设置样式 (针对 Rap 歌词优化的样式)
    # Alignment=2: 底部居中
    # Outline=1: 黑色描边，防止白色背景看不清
    # MarginV=20: 距离底部的垂直边距
    # Fontname: 优先使用苹方，Mac 自带，兼容性最好
    style = (
        "Fontname=PingFang SC,"
        "Fontsize=18,"
        "PrimaryColour=&H00FFFFFF,"  # 白色文字
        "OutlineColour=&H00000000,"  # 黑色描边
        "Outline=1,"
        "Shadow=0,"
        "Alignment=2,"
        "MarginV=25"
    )

    # 4. 构建命令
    # 重点：subtitles 滤镜的路径在 Mac 下如果包含空格，需要进行二次转义
    escaped_srt_path = abs_srt.replace(":", "\\:").replace("'", "'\\\\\\''")
    
    cmd = [
        'ffmpeg',
        '-i', abs_video,
        '-vf', f"subtitles='{escaped_srt_path}':force_style='{style}'",
        '-c:v', 'libx264',
        '-preset', 'medium',   # 兼顾速度与画质
        '-crf', '20',          # 高质量压制
        '-c:a', 'copy',        # 音频直接复制，不损失音质
        '-y',                  # 覆盖输出
        output_path
    ]

    print(f"🚀 开始压制字幕...")
    print(f"输入视频: {os.path.basename(abs_video)}")
    print(f"输入字幕: {os.path.basename(abs_srt)}")

    try:
        # 使用 subprocess.run 运行，并捕获错误信息
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"\n✅ 压制成功！")
            print(f"🎥 成品路径: {output_path}")
        else:
            print(f"\n❌ 压制失败！FFmpeg 报错如下:")
            print(result.stderr)
            
    except Exception as e:
        print(f"程序运行发生异常: {e}")

# --- 运行示例 ---
# if __name__ == "__main__":
video = "/Users/randy/Downloads/input.mp4"
srt = "/Users/randy/Downloads/input.srt"
merge_subtitles_to_video(video, srt)

🚀 开始压制字幕...
输入视频: input.mp4
输入字幕: input.srt

✅ 压制成功！
🎥 成品路径: /Users/randy/Downloads/final_rap_video.mp4
